## Binary classification with class imbalance

In [6]:
import torch
import torch.nn as nn


# --- SETUP DATA ---
# Let's say: 1000 total samples
# 900 are Background (Class 0), 100 are Foreground (Class 1)
num_neg = 900
num_pos = 100

# Targets: [Neg, Pos]
targets = torch.tensor([0.0, 1.0]) 
# Raw model outputs (logits)
logits = torch.tensor([0.8, 1.2]) 

# --- CALCULATE WEIGHT ---
# pos_weight = (count of negative) / (count of positive)
pos_weight = torch.tensor([num_neg / num_pos])

# --- PYTORCH NATIVE ---
# Note: pos_weight only scales the positive term in the BCE formula
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
pytorch_loss = criterion(logits, targets)

# --- MANUAL CALCULATION ---
# Formula: -[pos_weight * y * log(sigmoid(p)) + (1-y) * log(1-sigmoid(p))]
probs = torch.sigmoid(logits)
manual_loss = -(pos_weight * targets * torch.log(probs) + (1 - targets) * torch.log(1 - probs))
manual_loss_mean = manual_loss.mean()

print(f"Binary PyTorch Loss: {pytorch_loss.item():.4f}")
print(f"Binary Manual Loss:  {manual_loss_mean.item():.4f}")

Binary PyTorch Loss: 1.7703
Binary Manual Loss:  1.7703


## Multiclass classification with class imbalance

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Multiclass Classification with Class Imbalance

# --- SETUP DATA ---
# 3 Classes: Apple (rare), Banana (common), Orange (medium)
counts = torch.tensor([10, 80, 50]) # Sample counts per class
N = counts.sum()                    # Total samples
K = len(counts)                     # Number of classes

# Targets: 1 Apple (Index 0), 1 Banana (Index 1)
targets = torch.tensor([0, 1])
logits = torch.tensor([[1.5, 0.5, 0.2],  # Prediction for Apple
                       [0.1, 2.0, 0.5]]) # Prediction for Banana

# --- CALCULATE WEIGHTS ---
# Formula: alpha_c = N / (K * count_c)
weights = N / (K * counts)

# --- PYTORCH NATIVE ---
criterion = nn.CrossEntropyLoss(weight=weights)
pytorch_loss = criterion(logits, targets)

# --- MANUAL CALCULATION ---
# 1. Get Log Softmax
log_probs = F.log_softmax(logits, dim=1)

# 2. Extract log-probs for the true classes
target_log_probs = log_probs[torch.arange(len(targets)), targets]

# 3. Apply weights based on target class
sample_weights = weights[targets]
weighted_losses = -(target_log_probs * sample_weights)

# 4. Reduction (PyTorch defaults to weighted mean)
manual_loss_mean = weighted_losses.sum() / sample_weights.sum()

print(f"Multiclass PyTorch Loss: {pytorch_loss.item():.4f}")
print(f"Multiclass Manual Loss:  {manual_loss_mean.item():.4f}")

Multiclass PyTorch Loss: 0.4752
Multiclass Manual Loss:  0.4752
